# Kozhikode Rainfall Explorer
Fetch and visualize rainfall data for any date range from the Open-Meteo archive API.

In [2]:
# ── Date range — change these two values to explore any period ──
START_DATE = "2026-01-01"
END_DATE   = "2026-08-12"

In [3]:
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

LATITUDE  = 11.2588
LONGITUDE = 75.7804

params = {
    "latitude":   LATITUDE,
    "longitude":  LONGITUDE,
    "start_date": START_DATE,
    "end_date":   END_DATE,
    "daily":      ["precipitation_sum", "rain_sum", "temperature_2m_max", "temperature_2m_min"],
    "timezone":   "Asia/Kolkata"
}

resp = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=30)
resp.raise_for_status()

raw = resp.json()["daily"]
df = pd.DataFrame({
    "date":          pd.to_datetime(raw["time"]),
    "rainfall_mm":   raw["rain_sum"],
    "precipitation": raw["precipitation_sum"],
    "max_temp_c":    raw["temperature_2m_max"],
    "min_temp_c":    raw["temperature_2m_min"],
})

df["month"]     = df["date"].dt.strftime("%b %Y")
df["rain_cat"]  = pd.cut(
    df["rainfall_mm"],
    bins=[-1, 0, 10, 35, 65, 999],
    labels=["No rain", "Light", "Moderate", "Heavy", "Very heavy"]
)

print(f"Fetched {len(df)} days  |  {START_DATE} → {END_DATE}")
print(f"Total rainfall : {df['rainfall_mm'].sum():.1f} mm")
print(f"Rainy days     : {(df['rainfall_mm'] > 0).sum()}")
print(f"Max single day : {df['rainfall_mm'].max():.1f} mm  on {df.loc[df['rainfall_mm'].idxmax(), 'date'].date()}")

Fetched 224 days  |  2026-01-01 → 2026-08-12
Total rainfall : 2099.0 mm
Rainy days     : 190
Max single day : 145.9 mm  on 2026-08-01


## Data Table

In [4]:
display_df = df[["date", "rainfall_mm", "precipitation", "max_temp_c", "min_temp_c", "rain_cat"]].copy()
display_df["date"] = display_df["date"].dt.strftime("%Y-%m-%d")

display(
    display_df.style
        .format({"rainfall_mm": "{:.1f}", "precipitation": "{:.1f}",
                 "max_temp_c": "{:.1f}", "min_temp_c": "{:.1f}"})
        .background_gradient(subset=["rainfall_mm"], cmap="Blues")
        .set_caption(f"Daily weather data — Kozhikode ({START_DATE} to {END_DATE})")
)

,date,rainfall_mm,precipitation,max_temp_c,min_temp_c,rain_cat
0,2026-01-01,0.0,0.0,30.9,22.6,No rain
1,2026-01-02,0.3,0.3,31.2,23.3,Light
2,2026-01-03,1.0,1.0,30.1,22.4,Light
3,2026-01-04,0.0,0.0,32.3,22.1,No rain
4,2026-01-05,0.1,0.1,31.7,20.8,Light
5,2026-01-06,0.1,0.1,32.3,20.1,Light
6,2026-01-07,0.3,0.3,31.6,21.0,Light
7,2026-01-08,0.0,0.0,32.0,22.0,No rain
8,2026-01-09,0.0,0.0,31.6,20.1,No rain
9,2026-01-10,0.0,0.0,31.5,20.6,No rain


## Daily Rainfall Bar Chart

In [5]:
fig = px.bar(
    df, x="date", y="rainfall_mm",
    color="rain_cat",
    color_discrete_map={
        "No rain":    "#e0e0e0",
        "Light":      "#90caf9",
        "Moderate":   "#42a5f5",
        "Heavy":      "#1565c0",
        "Very heavy": "#0d0067"
    },
    labels={"rainfall_mm": "Rainfall (mm)", "date": "Date", "rain_cat": "Intensity"},
    title=f"Daily Rainfall — Kozhikode ({START_DATE} to {END_DATE})",
    template="plotly_dark"
)
fig.update_layout(xaxis_tickangle=-45, legend_title="Intensity")
fig.show()

## Monthly Rainfall Summary

In [6]:
monthly = (
    df.groupby(df["date"].dt.to_period("M"))
      .agg(total_rainfall=("rainfall_mm", "sum"),
           rainy_days=("rainfall_mm", lambda x: (x > 0).sum()),
           max_day=("rainfall_mm", "max"))
      .reset_index()
)
monthly["date"] = monthly["date"].dt.to_timestamp()

fig2 = go.Figure()
fig2.add_bar(x=monthly["date"], y=monthly["total_rainfall"],
             name="Total rainfall (mm)", marker_color="#42a5f5")
fig2.add_scatter(x=monthly["date"], y=monthly["max_day"],
                 mode="lines+markers", name="Max single day (mm)",
                 line=dict(color="#ff7043", width=2), marker=dict(size=7))
fig2.update_layout(
    title=f"Monthly Rainfall Summary — Kozhikode ({START_DATE} to {END_DATE})",
    xaxis_title="Month", yaxis_title="Rainfall (mm)",
    template="plotly_dark", legend=dict(orientation="h", y=1.1)
)
fig2.show()

display(
    monthly.rename(columns={"date": "Month", "total_rainfall": "Total (mm)",
                             "rainy_days": "Rainy Days", "max_day": "Max Day (mm)"})
           .style.format({"Total (mm)": "{:.1f}", "Max Day (mm)": "{:.1f}"})
           .background_gradient(subset=["Total (mm)"], cmap="Blues")
)

,Month,Total (mm),Rainy Days,Max Day (mm)
0,2026-01-01 00:00:00,38.5,19,15.0
1,2026-02-01 00:00:00,53.7,12,20.7
2,2026-03-01 00:00:00,100.1,29,16.6
3,2026-04-01 00:00:00,149.5,27,34.4
4,2026-05-01 00:00:00,278.8,31,23.5
5,2026-06-01 00:00:00,705.8,30,66.4
6,2026-07-01 00:00:00,412.5,30,48.4
7,2026-08-01 00:00:00,360.1,12,145.9


## 7-Day Rolling Average

In [7]:
df["rolling_7d"] = df["rainfall_mm"].rolling(7, min_periods=1).mean()

fig3 = go.Figure()
fig3.add_bar(x=df["date"], y=df["rainfall_mm"],
             name="Daily rainfall", marker_color="rgba(66,165,245,0.4)")
fig3.add_scatter(x=df["date"], y=df["rolling_7d"],
                 mode="lines", name="7-day rolling avg",
                 line=dict(color="#ff7043", width=2))
fig3.update_layout(
    title="Daily Rainfall with 7-Day Rolling Average",
    xaxis_title="Date", yaxis_title="Rainfall (mm)",
    template="plotly_dark"
)
fig3.show()

## June – August Rainfall Line Chart

In [8]:
jun_aug = df[df["date"].dt.month.isin([6, 7, 8])].copy()

fig4 = px.line(
    jun_aug, x="date", y="rainfall_mm",
    labels={"rainfall_mm": "Rainfall (mm)", "date": "Date"},
    title="Daily Rainfall — June to August",
    template="plotly_dark"
)
fig4.update_traces(line_color="#42a5f5")
fig4.show()

## Last Year — June to August (Comparison)

In [9]:
import requests, pandas as pd, plotly.graph_objects as go

start_year = pd.to_datetime(START_DATE).year
last_year  = start_year - 1

params_ly = {
    "latitude":   LATITUDE,
    "longitude":  LONGITUDE,
    "start_date": f"{last_year}-06-01",
    "end_date":   f"{last_year}-08-31",
    "daily":      ["rain_sum"],
    "timezone":   "Asia/Kolkata"
}
raw_ly = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params_ly, timeout=30).json()["daily"]
df_ly = pd.DataFrame({"date": pd.to_datetime(raw_ly["time"]), "rainfall_mm": raw_ly["rain_sum"]})

fig5 = go.Figure()
fig5.add_scatter(x=jun_aug["date"], y=jun_aug["rainfall_mm"],
                 mode="lines", name=f"Jun–Aug {start_year}",
                 line=dict(color="#42a5f5", width=2))
# align last year's dates to this year for side-by-side comparison on the same x-axis
df_ly_aligned = df_ly.copy()
df_ly_aligned["date"] = df_ly_aligned["date"] + pd.DateOffset(years=1)
fig5.add_scatter(x=df_ly_aligned["date"], y=df_ly_aligned["rainfall_mm"],
                 mode="lines", name=f"Jun–Aug {last_year}",
                 line=dict(color="#ff7043", width=2, dash="dash"))
fig5.update_layout(
    title=f"Jun–Aug Rainfall: {start_year} vs {last_year}",
    xaxis_title="Date", yaxis_title="Rainfall (mm)",
    template="plotly_dark"
)
fig5.show()

print(f"\n{start_year} Jun–Aug total : {jun_aug['rainfall_mm'].sum():.1f} mm")
print(f"{last_year} Jun–Aug total  : {df_ly['rainfall_mm'].sum():.1f} mm")


2026 Jun–Aug total : 1478.4 mm
2025 Jun–Aug total  : 1616.2 mm
